# Reranking em Python com Elasticsearch Basic

Continuação dos tutoriais de BM25, semântica e busca híbrida. Usamos os documentos já indexados e a rota `/search/hybrid`:

```text
BM25 + kNN → fusão RRF da API → N candidatos → cross-encoder em Python → K melhores
```

A estratégia de avaliar cada par **pergunta–trecho** e reordenar os candidatos é semelhante à do `text_similarity_reranker`. Aqui o modelo roda no processo Python do notebook, enquanto a recuperação continua usando Elasticsearch Basic.

Pré-requisitos: API RAG e embeddings em execução, acervo já indexado e dependências instaladas no ambiente do kernel. Na raiz do projeto:

```bash
pip install -e '.[notebook,reranking]'
```

Usaremos o modelo [mmarco-mMiniLMv2-L12-H384-v1](https://huggingface.co/cross-encoder/mmarco-mMiniLMv2-L12-H384-v1), um cross-encoder multilíngue para recuperação de informação.

## 1. Configurar a consulta

N define quantos candidatos recuperamos; K define quantos retornamos. Altere a pergunta e os filtros para seu acervo. A API limita N por `search.max_top_k`.

In [8]:
from pathlib import Path
import math
import httpx
import pandas as pd
import yaml
from IPython.display import display

RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config/rag.yaml").exists()), None)

CONFIG_PATH = RAIZ / "config/rag.local.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text())

API_URL = "http://localhost:8000"
PERGUNTA = "quais erros aparecem antes da estabilização térmica?"
FILTROS = {}  
N, K = 30, 5
MODELO_RERANK = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
DEVICE = "cpu"  
MAX_LENGTH = 512  
BATCH_SIZE = 4
assert 1 <= K <= N

headers = {}
auth = CONFIG.get("api", {}).get("auth", {})
if auth.get("enabled"):
    headers["Authorization"] = f"Bearer {auth['token']}"
print({"configuração": str(CONFIG_PATH), "candidatos": N, "top_k": K, "device": DEVICE})

{'configuração': '/home/golo/Documentos/websensors-rag/config/rag.local.yaml', 'candidatos': 30, 'top_k': 5, 'device': 'cpu'}


## 2. Recuperar os candidatos uma única vez

A API calcula o embedding da pergunta, faz BM25/kNN e aplica RRF na aplicação. Os candidatos abaixo são exatamente os que serão avaliados pelo reranker; nenhuma outra busca é feita durante a comparação.

In [9]:
with httpx.Client(base_url=API_URL, headers=headers, timeout=120) as api:
    descricao = api.get("/rag")
    limite_api = descricao.json().get("search", {}).get("max_top_k", CONFIG.get("search", {}).get("max_top_k", 50))
    resposta = api.post("/search/hybrid", json={"query": PERGUNTA, "top_k": N, "filters": FILTROS, "debug": True})
    recuperacao = resposta.json()
candidatos = recuperacao["results"]


df_exe = pd.DataFrame([{"posição_RRF": h["rank"], "chunk_id": h["chunk_id"], "score_RRF": h["score"], "trecho": h["content"][:250]} for h in candidatos])
df_exe

,posição_RRF,chunk_id,score_RRF,trecho
0,1,pira2-aa6e73614ce20a79a63c:bb7342474da08036ada...,0.030622,A fadiga é uma das principais condições de gov...
1,2,pira2-09774092ebc8d8b9de51:4340e94c3cf6cc51de5...,0.030579,Vários eventos de influxo de gás ocorreram dur...
2,3,pira2-fd001576d3f95b8fd6d3:df89c19af3c88785e93...,0.030090,Piercement de diapir através do assoalho do oc...
3,4,pira2-1bc9c0296cd8642518a9:69abf31f46c738135fc...,0.029727,Impactos das mudanças climáticas e mudanças re...
4,5,pira2-5b6f5fc44409118667b4:6d4afd8a822dfecd264...,0.027444,"Em 16 de julho de 2000, um derramamento de pet..."
5,6,pira2-3081720c8fff426282cf:7fbf7111adf68edc16b...,0.025383,No contexto do rio Doce (Sudeste Brasil) Fundã...
6,7,pira2-0ec8e2d026399e8fc1dc:2582fd87b2e4e83626b...,0.025253,"As zonas salgadas de perfuração, especialmente..."
7,8,pira2-51964ab16a8b666394ab:c94787ec4c9196b4261...,0.024837,A atual crise financeira global e outras quest...
8,9,pira2-336ce2d901ec1d29cf78:8e246112408ed880834...,0.024100,As margens passivas subjacentes por um desapeg...
9,10,pira2-c3e70aa6ed9a4bfe2911:1b86b08615047aa4221...,0.024093,Coquinas constituem depósitos generalizados em...


## 3. Carregar o reranker

In [10]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(MODELO_RERANK, device=DEVICE, max_length=MAX_LENGTH)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

## 4. Avaliar pergunta–trecho e selecionar top-k

A função preserva os candidatos originais, guarda a posição/score RRF e ordena pelo score do cross-encoder. Empates mantêm a ordem anterior. Nenhum score RRF é somado ao score do modelo.

In [11]:
def reranquear(pergunta, trechos, top_k):
    pares = [(pergunta, trecho["content"]) for trecho in trechos]
    scores = reranker.predict(pares, batch_size=BATCH_SIZE, show_progress_bar=False)
    avaliados = []
    for trecho, score in zip(trechos, scores):
        score = float(score)
        avaliados.append({**trecho, "rank_rrf": trecho["rank"], "score_rrf": trecho["score"], "reranker_score": score})
    avaliados.sort(key=lambda trecho: trecho["reranker_score"], reverse=True)
    return [{**trecho, "rank": i} for i, trecho in enumerate(avaliados[:top_k], 1)]

resultados = reranquear(PERGUNTA, candidatos, K)
resultados

[{'rank': 1,
  'document_id': 'pira2-3081720c8fff426282cf',
  'chunk_id': 'pira2-3081720c8fff426282cf:7fbf7111adf68edc16bb4d2e:000001',
  'revision_id': '7fbf7111adf68edc16bb4d2e',
  'filename': 'pira2-3081720c8fff426282cf.md',
  'title': 'Texto de apoio Pirá 077',
  'section': 'Texto de apoio Pirá 077',
  'heading_path': ['Texto de apoio Pirá 077'],
  'page_start': None,
  'page_end': None,
  'content': 'No contexto do rio Doce (Sudeste Brasil) Fundão Darragem em 2015, monitoramos as mudanças nas concentrações de metal (loid) s em água e sedimentos e seus particionamentos particulados e dissolvidos ao longo do tempo. As amostras foram coletadas antes, durante, e após a chegada dos rejeitos da mina para o estuário do rio Doce (pré-impacto: 12, 10, 3 e 1 dia; Estágio agudo: Dia de volta - TD e 1 dia após - DA; Estágio crônico: 3 meses e 1 ano pós-desastre). Nossos resultados mostram que as concentrações de metal (loid) aumentaram significativamente com o tempo após o desastre e mudaram 

## 5. Comparar as posições e ler as evidências

Uma diferença positiva em `subiu_posições` indica que o trecho subiu. Os scores RRF e reranker têm escalas diferentes e não devem ser comparados diretamente nem tratados como probabilidades. Julgue se o texto responde melhor à pergunta.

In [13]:
comparacao = pd.DataFrame([
    {"posição_RRF": h["rank_rrf"], "posição_rerank": h["rank"],
     "subiu_posições": h["rank_rrf"] - h["rank"], "chunk_id": h["chunk_id"],
     "score_RRF": h["score_rrf"], "score_reranker": h["reranker_score"], "trecho": h["content"][:400]}
    for h in resultados
])
display(comparacao)
print("Top-k original:", [h["chunk_id"] for h in candidatos[:K]])
print("Top-k após reranking:", [h["chunk_id"] for h in resultados])
for h in resultados:
    print(f"\n{h['rank']}. {h['chunk_id']} — posição anterior: {h['rank_rrf']}\n{h['content']}")

,posição_RRF,posição_rerank,subiu_posições,chunk_id,score_RRF,score_reranker,trecho
0,6,1,5,pira2-3081720c8fff426282cf:7fbf7111adf68edc16b...,0.025383,-3.706393,No contexto do rio Doce (Sudeste Brasil) Fundã...
1,11,2,9,pira2-2c274693bc215941fc74:e076e8284346bf076dc...,0.023985,-3.772947,O mapeamento de sistemas de hidratos de gás na...
2,1,3,-2,pira2-aa6e73614ce20a79a63c:bb7342474da08036ada...,0.030622,-3.792816,A fadiga é uma das principais condições de gov...
3,9,4,5,pira2-336ce2d901ec1d29cf78:8e246112408ed880834...,0.024100,-4.040123,As margens passivas subjacentes por um desapeg...
4,28,5,23,pira2-cd96e70a406c5deb8da3:3509bc2e3b5b457286f...,0.015873,-4.372966,O aquecimento do oceano e o morrimento de gelo...


Top-k original: ['pira2-aa6e73614ce20a79a63c:bb7342474da08036ada98ac5:000001', 'pira2-09774092ebc8d8b9de51:4340e94c3cf6cc51de5cdc41:000001', 'pira2-fd001576d3f95b8fd6d3:df89c19af3c88785e93c40fb:000001', 'pira2-1bc9c0296cd8642518a9:69abf31f46c738135fc75525:000001', 'pira2-5b6f5fc44409118667b4:6d4afd8a822dfecd264adc73:000001']
Top-k após reranking: ['pira2-3081720c8fff426282cf:7fbf7111adf68edc16bb4d2e:000001', 'pira2-2c274693bc215941fc74:e076e8284346bf076dc76613:000001', 'pira2-aa6e73614ce20a79a63c:bb7342474da08036ada98ac5:000001', 'pira2-336ce2d901ec1d29cf78:8e246112408ed880834eecde:000001', 'pira2-cd96e70a406c5deb8da3:3509bc2e3b5b457286f1e291:000001']

1. pira2-3081720c8fff426282cf:7fbf7111adf68edc16bb4d2e:000001 — posição anterior: 6
No contexto do rio Doce (Sudeste Brasil) Fundão Darragem em 2015, monitoramos as mudanças nas concentrações de metal (loid) s em água e sedimentos e seus particionamentos particulados e dissolvidos ao longo do tempo. As amostras foram coletadas antes, dur

## 6. Comparar no conjunto de teste

Usamos as mesmas perguntas do benchmark Pirá e comparamos **Recall@10, MRR@10 e nDCG@10** da busca híbrida com e sem reranking. Os candidatos são recuperados uma única vez por pergunta e agrupados por documento antes do cálculo das métricas.


In [14]:
import hashlib
import json
from collections import defaultdict

def metricas_test(trechos, relevantes):
    docs = list(dict.fromkeys(h["document_id"] for h in trechos))[:k_test]
    recall = len(set(docs) & relevantes) / len(relevantes)
    rr = next((1 / i for i, doc in enumerate(docs, 1) if doc in relevantes), 0.0)
    dcg = sum(1 / math.log2(i + 1) for i, doc in enumerate(docs, 1) if doc in relevantes)
    ideal = sum(1 / math.log2(i + 1) for i in range(1, min(k_test, len(relevantes)) + 1))
    return {"recall": recall, "rr": rr, "ndcg": dcg / ideal}

PASTA_BENCHMARK = RAIZ / "notebooks/resultados_pira2"
config_benchmark = json.loads((PASTA_BENCHMARK / "configuracao.json").read_text())
linhas_test = json.loads((RAIZ / "notebooks/.cache/pira2/test.json").read_text())
consultas_salvas = pd.read_csv(PASTA_BENCHMARK / "resultados_consultas.csv", dtype={"query_id": str})
ids_avaliados = consultas_salvas.loc[consultas_salvas["metodo"] == "Híbrida", "query_id"].drop_duplicates()

perguntas_por_id = {}
gabarito_test = defaultdict(set)
for linha in linhas_test:
    query_id = str(linha["id_qa"])
    if query_id in set(ids_avaliados):
        texto = linha["abstract_translated_pt"].strip()
        doc_id = "pira2-" + hashlib.sha256(texto.encode("utf-8")).hexdigest()[:20]
        perguntas_por_id[query_id] = linha["question_pt_origin"].strip()
        gabarito_test[query_id].add(doc_id)

k_test = config_benchmark["k_avaliacao"]
n_test = config_benchmark["top_k_api"]
filtros_test = {"colecao": config_benchmark["colecao"]}

registros_test = []
with httpx.Client(base_url=API_URL, headers=headers, timeout=120) as api_test:
    for query_id in ids_avaliados:
        pergunta = perguntas_por_id[query_id]
        resposta = api_test.post("/search/hybrid", json={"query": pergunta, "top_k": n_test, "filters": filtros_test})
        trechos = resposta.json()["results"]
        reordenados = reranquear(pergunta, trechos, len(trechos))
        for metodo, ranking in [("Híbrida", trechos), ("Híbrida + reranking", reordenados)]:
            registros_test.append({"metodo": metodo, **metricas_test(ranking, gabarito_test[query_id])})

DF_TESTE_RERANK = pd.DataFrame(registros_test)
RESUMO_TESTE_RERANK = DF_TESTE_RERANK.groupby("metodo", sort=False).agg(
    recall=("recall", "mean"), mrr=("rr", "mean"), ndcg=("ndcg", "mean"),
).rename(columns={"recall": f"Recall@{k_test}", "mrr": f"MRR@{k_test}", "ndcg": f"nDCG@{k_test}"})
RESUMO_TESTE_RERANK.round(4)


,Recall@10,MRR@10,nDCG@10
metodo,,,
Híbrida,0.9667,0.8706,0.8935
Híbrida + reranking,1.0000,0.9111,0.9341
